In [3]:
# ================================================================
# SigLIP-2 + ViLT Cross-Attention Fusion — Fakeddit 2-Way
# TARGET: 96%+ Accuracy | Overfitting Gap < 0.03
#
# DIAGNOSIS OF PREVIOUS VERSION (74% accuracy):
#  ❌ MAX_SAMPLES=5000 → only ~3200 train samples/fold (far too few)
#  ❌ Backbones fully FROZEN until epoch 8 → head-only training 40% of run
#  ❌ Only last 2 layers unfrozen → limited representational capacity
#  ❌ CosineAnnealingWarmRestarts with T_0=5 → LR spikes every 5 epochs
#     destroy fine-tuned weights, causing instability
#  ❌ Optimizer rebuilt at epoch 8 → Adam momentum state lost
#  ❌ SigLIP & ViLT use CLS token ONLY → discards rich patch/token features
#  ❌ DropPath on seq_len=1 ViLT vector → kills entire residual branch
#  ❌ No differential LR (backbone vs head same rate)
#  ❌ No label smoothing in FocalLoss → overconfident predictions
#  ❌ Early stopping on val_acc not val_F1 → unreliable convergence signal
#
# FIXES:
#  ✅ MAX_SAMPLES=20000 (4× more data, balanced)
#  ✅ Unfreeze last 4 SigLIP + last 4 ViLT blocks FROM EPOCH 1
#  ✅ Differential LR: backbone=5e-6, head=2e-4
#  ✅ Single cosine schedule with 15% warm-up (no restarts)
#  ✅ AttentionPooling over ALL patch/token sequences (not just CLS)
#  ✅ DropPath fixed: shape matches tensor rank correctly
#  ✅ Deeper cross-attention: 3 bidirectional layers with residuals
#  ✅ Label smoothing 0.10 inside FocalLoss
#  ✅ R-Drop consistency regularisation (dual forward KL)
#  ✅ Early stopping on val macro-F1, patience=7
#  ✅ SWA starts at 80% of training
#  ✅ TTA: 3 augmented passes on test set
#  ✅ Stronger image aug: RandAugment + RandomErasing + geometric
#  ✅ Per-class F1, confusion matrix, full classification_report
# ================================================================

# ── Cell 1: Install dependencies ─────────────────────────────────
import subprocess, sys

def pip_q(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_q('transformers', 'datasets', 'Pillow', 'scikit-learn',
      'pandas', 'numpy', 'timm', 'tqdm')

import os, random, warnings, math, copy, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoProcessor, AutoModel,
    ViltProcessor, ViltModel,
    get_cosine_schedule_with_warmup,
)
from PIL import Image, ImageEnhance, ImageOps, ImageFilter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import torchvision.transforms as T
import glob as _glob
import urllib.request as _urlreq
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Cell 2: Config ────────────────────────────────────────────────
SEED         = 42
NUM_CLASSES  = 2
LABEL_COL    = '2_way_label'
SIGLIP2_NAME = 'google/siglip-base-patch16-224'
VILT_NAME    = 'dandelin/vilt-b32-mlm'

# ── Training hyper-params ─────────────────────────────────────────
NUM_EPOCHS   = 30          # enough for SWA + early stopping
BATCH_SIZE   = 16          # per-device; eff. batch = 16×4 = 64
GRAD_ACCUM   = 4
LR_BACKBONE  = 5e-6        # conservative for pretrained blocks
LR_HEAD      = 2e-4        # aggressive for fresh projection + classifier
WEIGHT_DECAY = 1e-2
WARMUP_FRAC  = 0.15
N_FOLDS      = 5
MAX_SAMPLES  = 20_000      # 4× previous (10k/class → balanced)
D_MODEL      = 512
NUM_HEADS    = 8
N_CROSS_LAYERS = 3         # bidirectional cross-attention layers
DROPOUT      = 0.20
DROPPATH_P   = 0.08
LABEL_SMOOTH = 0.10
FOCAL_GAMMA  = 2.0
EMA_DECAY    = 0.9995
PATIENCE     = 7           # patience on val macro-F1
SWA_START    = 0.80        # start SWA at 80% of training
UNFREEZE_N   = 4           # unfreeze last N blocks from epoch 1
MIXUP_ALPHA  = 0.20
TTA_PASSES   = 3
USE_AMP      = True

OUTPUT_DIR   = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Config : epochs={NUM_EPOCHS} | batch={BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} eff '
      f'| LR bb={LR_BACKBONE} head={LR_HEAD} | samples={MAX_SAMPLES}')
print('='*72)

# ── Cell 3: Data loading (same Kaggle dataset, 4× more samples) ───
def _find_tsv():
    hits = _glob.glob('/kaggle/input/**/multimodal_train.tsv', recursive=True)
    if hits:
        return sorted(hits)[0]
    hits = [p for p in _glob.glob('/kaggle/input/**/*.tsv', recursive=True)
            if 'train' in os.path.basename(p).lower()]
    return sorted(hits)[0] if hits else None

_tsv = _find_tsv()
if _tsv is None:
    raise FileNotFoundError(
        'multimodal_train.tsv not found.\n'
        "Add Fakeddit: Notebook → Add data → 'vanshikavmittal fakeddit-dataset'."
    )
TSV_PATH = _tsv
DATA_DIR  = os.path.dirname(TSV_PATH)
print(f'TSV: {TSV_PATH}')

df_raw = pd.read_csv(TSV_PATH, sep='\t')
print(f'Raw rows: {len(df_raw)} | Cols: {df_raw.columns.tolist()}')

if 'image_url' not in df_raw.columns:
    raise KeyError("TSV has no 'image_url' column — cannot download images.")

df_raw = df_raw[
    df_raw['image_url'].notna() &
    (df_raw['image_url'].str.strip() != '')
].reset_index(drop=True)
print(f'Rows with image_url: {len(df_raw)}')

_le_pre = LabelEncoder()
df_raw['_lbl'] = _le_pre.fit_transform(df_raw[LABEL_COL])

df_sampled = (
    df_raw.groupby('_lbl', group_keys=False)
          .apply(lambda g: g.sample(
              min(len(g), MAX_SAMPLES // NUM_CLASSES),
              random_state=SEED))
          .reset_index(drop=True)
)
df_sampled = df_sampled.drop(columns=['_lbl'])
print(f'Sampled to download: {len(df_sampled)}')

IMG_DIR = os.path.join(OUTPUT_DIR, 'images')
os.makedirs(IMG_DIR, exist_ok=True)

_HEADERS = {'User-Agent': 'Mozilla/5.0'}
_TIMEOUT = 10

def _download(row):
    dest = os.path.join(IMG_DIR, f"{row['id']}.jpg")
    if os.path.isfile(dest):
        return row['id'], True
    try:
        req = _urlreq.Request(str(row['image_url']), headers=_HEADERS)
        with _urlreq.urlopen(req, timeout=_TIMEOUT) as resp:
            data = resp.read()
        if len(data) < 500:
            return row['id'], False
        with open(dest, 'wb') as fh:
            fh.write(data)
        return row['id'], True
    except Exception:
        return row['id'], False

print(f'Downloading up to {len(df_sampled)} images …')
_rows   = [row for _, row in df_sampled.iterrows()]
_ok_ids = set()

with ThreadPoolExecutor(max_workers=32) as pool:
    futs = {pool.submit(_download, r): r['id'] for r in _rows}
    for fut in tqdm(as_completed(futs), total=len(futs), desc='Images'):
        _id, ok = fut.result()
        if ok:
            _ok_ids.add(_id)

print(f'Downloaded: {len(_ok_ids)} / {len(df_sampled)}')
df_dl = df_sampled[df_sampled['id'].isin(_ok_ids)].reset_index(drop=True)
df_dl['img_path'] = df_dl['id'].apply(lambda x: os.path.join(IMG_DIR, f'{x}.jpg'))
print(f'Rows with images: {len(df_dl)}')

if len(df_dl) == 0:
    raise RuntimeError(
        '0 images downloaded.\n'
        'Enable Internet in Notebook settings (⚙ → Internet → ON).'
    )

# Re-encode labels on downloaded subset first
le = LabelEncoder()
df_dl['label_enc'] = le.fit_transform(df_dl[LABEL_COL])
print('Classes:', le.classes_)
print(f'Before rebalance:\n{df_dl["label_enc"].value_counts().to_string()}')

# Re-balance AFTER download — download success rate differs by class
# (e.g. 8149 class-0 vs 3589 class-1 in run logs → must fix before training)
_vc      = df_dl['label_enc'].value_counts()
_min_cnt = int(_vc.min())
df = (
    df_dl.groupby('label_enc', group_keys=False)
         .apply(lambda g: g.sample(_min_cnt, random_state=SEED))
         .sample(frac=1, random_state=SEED)
         .reset_index(drop=True)
)
print(f'After rebalance:\n{df["label_enc"].value_counts().to_string()}')
print(f'Final balanced rows: {len(df)}')

# ── Cell 4: Image Augmentation ────────────────────────────────────
_CLIP_MEAN = (0.48145466, 0.4578275,  0.40821073)
_CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

def _build_train_aug():
    return T.Compose([
        T.RandomResizedCrop(224, scale=(0.80, 1.0), ratio=(0.9, 1.1)),
        T.RandomHorizontalFlip(0.5),
        T.RandomVerticalFlip(0.05),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.08),
        T.RandomGrayscale(p=0.05),
        T.RandomRotation(15),
        T.ToTensor(),
        T.Normalize(_CLIP_MEAN, _CLIP_STD),
        T.RandomErasing(p=0.20, scale=(0.02, 0.15)),
    ])

def _build_val_aug():
    return T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(_CLIP_MEAN, _CLIP_STD),
    ])

def _build_tta_aug():
    return T.Compose([
        T.RandomResizedCrop(224, scale=(0.88, 1.0)),
        T.RandomHorizontalFlip(0.5),
        T.ToTensor(),
        T.Normalize(_CLIP_MEAN, _CLIP_STD),
    ])

TRAIN_AUG = _build_train_aug()
VAL_AUG   = _build_val_aug()
TTA_AUG   = _build_tta_aug()

def augment_text(text: str, p_drop: float = 0.08, p_swap: float = 0.05) -> str:
    """Word dropout + adjacent swap (train-only)."""
    _STOP = {'the','a','an','is','was','are','were','be','been','have','has',
             'do','does','did','will','would','could','should','to','of',
             'in','for','on','with','at','by','from','and','but','or'}
    words = text.split()
    if not words:
        return text
    kept = [w for w in words if w.lower() in _STOP or random.random() > p_drop]
    words = kept if kept else words
    for i in range(len(words) - 1):
        if random.random() < p_swap:
            words[i], words[i+1] = words[i+1], words[i]
    return ' '.join(words)

# ── Cell 5: Dataset ───────────────────────────────────────────────
class FakedditDataset(Dataset):
    """
    Returns (sig_inputs, vilt_inputs, label).
    Images: real augmented image tensor (aug depends on mode).
    Processors: siglip_proc handles image pre-processing natively;
    we override pixel_values with our augmented tensor.
    """
    def __init__(self, df, siglip_proc, vilt_proc, mode: str = 'train'):
        self.df          = df.reset_index(drop=True)
        self.siglip_proc = siglip_proc
        self.vilt_proc   = vilt_proc
        self.mode        = mode
        self.aug = {'train': TRAIN_AUG, 'tta': TTA_AUG}.get(mode, VAL_AUG)
        self.placeholder = Image.new('RGB', (224, 224), (128, 128, 128))

    def __len__(self):
        return len(self.df)

    def _load_img(self, path):
        try:
            return Image.open(path).convert('RGB')
        except Exception:
            return self.placeholder.copy()

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label_enc'])
        text  = str(row.get('clean_title', row.get('title', '')))[:256]

        if self.mode == 'train':
            text = augment_text(text)

        img_pil = self._load_img(row['img_path'])
        # Augmented tensor for direct use
        img_tensor = self.aug(img_pil)   # (3, 224, 224)

        # SigLIP: let processor handle tokenisation; override pixel_values
        sig_in = self.siglip_proc(
            text=[text], images=img_pil,
            return_tensors='pt', padding='max_length',
            truncation=True, max_length=64,
        )
        sig_in = {k: v.squeeze(0) for k, v in sig_in.items()}
        sig_in['pixel_values'] = img_tensor   # use augmented version

        # ViLT — CRITICAL FIX: resize to exactly 224x224 before processor
        # ViLT's ViltImageProcessor resizes based on aspect ratio by default,
        # producing variable shapes like (3,352,608) or (3,384,512) which
        # cannot be stacked in a batch → RuntimeError in collate_fn.
        # Fix: pass a PIL image already at 224×224 so processor outputs (3,224,224).
        img_vilt = img_pil.resize((224, 224), Image.BILINEAR)
        vilt_in = self.vilt_proc(
            text=[text], images=img_vilt,
            return_tensors='pt', padding='max_length',
            truncation=True, max_length=40,
        )
        vilt_in = {k: v.squeeze(0) for k, v in vilt_in.items()}
        # Verify shape is fixed — if processor still returns wrong shape, override
        if vilt_in['pixel_values'].shape[-1] != 224 or vilt_in['pixel_values'].shape[-2] != 224:
            # Fallback: use our own VILT-normalised tensor at 224x224
            _VILT_MEAN = (0.5, 0.5, 0.5)
            _VILT_STD  = (0.5, 0.5, 0.5)
            _fix = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                              T.Normalize(_VILT_MEAN, _VILT_STD)])
            vilt_in['pixel_values'] = _fix(img_pil)

        return sig_in, vilt_in, label


def collate_fn(batch):
    sig_list, vilt_list, labels = zip(*batch)
    def stack(lst):
        return {k: torch.stack([d[k] for d in lst]) for k in lst[0]}
    return stack(sig_list), stack(vilt_list), torch.tensor(labels, dtype=torch.long)

print('✅ Dataset ready')

# ── Cell 6: Model Architecture ────────────────────────────────────

class StochasticDepth(nn.Module):
    """
    Drop the entire residual with probability p.
    Shape-safe: expands keep-mask to match all dims of x.
    """
    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0.0:
            return x
        # (B, 1, 1, ...) broadcasts over seq and feature dims
        shape = [x.shape[0]] + [1] * (x.dim() - 1)
        keep  = (torch.rand(shape, device=x.device) > self.p).float()
        return x * keep


class AttentionPool(nn.Module):
    """
    Learnable attention pooling over a sequence.
    Pools ALL tokens (not just CLS), weighted by a learned score.
    """
    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, S, D) → (B, D)
        w = torch.softmax(self.score(x), dim=1)   # (B, S, 1)
        return (x * w).sum(dim=1)                  # (B, D)


class CrossAttentionLayer(nn.Module):
    """
    One bidirectional cross-attention layer with FFN and residuals.
    x attends to y; y attends to x.  Both use pre-norm (more stable).
    """
    def __init__(self, d_model: int, n_heads: int,
                 dropout: float = 0.10, sd_p: float = 0.08):
        super().__init__()
        # x → y
        self.norm_x1 = nn.LayerNorm(d_model)
        self.attn_xy = nn.MultiheadAttention(d_model, n_heads,
                                              dropout=dropout, batch_first=True)
        self.sd_xy   = StochasticDepth(sd_p)
        self.norm_x2 = nn.LayerNorm(d_model)
        self.ffn_x   = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )
        self.sd_fx   = StochasticDepth(sd_p)

        # y → x
        self.norm_y1 = nn.LayerNorm(d_model)
        self.attn_yx = nn.MultiheadAttention(d_model, n_heads,
                                              dropout=dropout, batch_first=True)
        self.sd_yx   = StochasticDepth(sd_p)
        self.norm_y2 = nn.LayerNorm(d_model)
        self.ffn_y   = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )
        self.sd_fy   = StochasticDepth(sd_p)

    def forward(self, x: torch.Tensor,
                y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # Pre-norm cross-attention: x queries y
        x_norm   = self.norm_x1(x)
        y_norm_k = self.norm_y1(y)
        ax, _    = self.attn_xy(x_norm, y_norm_k, y_norm_k)
        x        = x + self.sd_xy(ax)
        x        = x + self.sd_fx(self.ffn_x(self.norm_x2(x)))

        # Pre-norm cross-attention: y queries x
        y_norm   = self.norm_y1(y)
        x_norm_k = self.norm_x1(x)
        ay, _    = self.attn_yx(y_norm, x_norm_k, x_norm_k)
        y        = y + self.sd_yx(ay)
        y        = y + self.sd_fy(self.ffn_y(self.norm_y2(y)))

        return x, y


class SigLIP2Encoder(nn.Module):
    """
    Encodes image + text via SigLIP-2.
    Uses AttentionPool over ALL vision & text tokens (not just CLS).
    Returns two sequences for cross-attention:
      vis_seq : (B, 1, D)  — pooled vision embedding
      txt_seq : (B, 1, D)  — pooled text  embedding
    """
    def __init__(self, model, d_model: int, unfreeze_n: int):
        super().__init__()
        self.model = model

        # Freeze all, then selectively unfreeze
        for p in self.model.parameters():
            p.requires_grad = False

        # Unfreeze last unfreeze_n vision blocks
        vis_layers = self.model.vision_model.encoder.layers
        for layer in vis_layers[-unfreeze_n:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in self.model.vision_model.post_layernorm.parameters():
            p.requires_grad = True

        # Unfreeze last unfreeze_n text blocks
        txt_layers = self.model.text_model.encoder.layers
        for layer in txt_layers[-unfreeze_n:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in self.model.text_model.final_layer_norm.parameters():
            p.requires_grad = True

        vis_h = model.config.vision_config.hidden_size   # 768
        txt_h = model.config.text_config.hidden_size     # 768

        self.proj_vis = nn.Sequential(
            nn.Linear(vis_h, d_model), nn.LayerNorm(d_model), nn.GELU(),
            nn.Dropout(0.10),
        )
        self.proj_txt = nn.Sequential(
            nn.Linear(txt_h, d_model), nn.LayerNorm(d_model), nn.GELU(),
            nn.Dropout(0.10),
        )
        self.pool_vis = AttentionPool(d_model)
        self.pool_txt = AttentionPool(d_model)

    def forward(self, inputs: dict) -> tuple[torch.Tensor, torch.Tensor]:
        out = self.model(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs.get('attention_mask'),
            pixel_values   = inputs['pixel_values'],
        )
        # ALL vision tokens (B, S_v, vis_h)
        vis_tokens = self.proj_vis(
            out.vision_model_output.last_hidden_state)   # (B, S_v, D)
        # ALL text tokens (B, S_t, txt_h)
        txt_tokens = self.proj_txt(
            out.text_model_output.last_hidden_state)     # (B, S_t, D)

        vis_pooled = self.pool_vis(vis_tokens).unsqueeze(1)  # (B, 1, D)
        txt_pooled = self.pool_txt(txt_tokens).unsqueeze(1)  # (B, 1, D)
        return vis_pooled, txt_pooled


class ViLTEncoder(nn.Module):
    """
    Encodes multimodal joint representation via ViLT.
    Uses AttentionPool over ALL output tokens (not just CLS).
    Returns: (B, S_vilt, D) sequence for cross-attention.
    """
    def __init__(self, model, d_model: int, unfreeze_n: int):
        super().__init__()
        self.model = model

        for p in self.model.parameters():
            p.requires_grad = False

        # Unfreeze last unfreeze_n ViLT transformer blocks
        enc_layers = self.model.encoder.layer
        for layer in enc_layers[-unfreeze_n:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in self.model.layernorm.parameters():
            p.requires_grad = True
        if hasattr(self.model, 'pooler'):
            for p in self.model.pooler.parameters():
                p.requires_grad = True

        hidden = model.config.hidden_size   # 768

        self.proj = nn.Sequential(
            nn.Linear(hidden, d_model), nn.LayerNorm(d_model), nn.GELU(),
            nn.Dropout(0.10),
        )
        self.pool = AttentionPool(d_model)

    def forward(self, inputs: dict) -> torch.Tensor:
        out = self.model(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs.get('attention_mask'),
            token_type_ids = inputs.get('token_type_ids'),
            pixel_values   = inputs['pixel_values'],
            pixel_mask     = inputs.get('pixel_mask'),
        )
        # ALL tokens (B, S, hidden)
        tokens = self.proj(out.last_hidden_state)   # (B, S, D)
        pooled = self.pool(tokens).unsqueeze(1)     # (B, 1, D)
        return pooled


class SigLIPViLTClassifier(nn.Module):
    """
    SigLIP-2 (vis + text) × ViLT → N-layer bidirectional cross-attention
    → attention pooling → concat → MLP → 2-class logits.

    Feature streams (all D-dim after projection):
      vis : SigLIP visual pooled  (B, 1, D)
      txt : SigLIP text pooled    (B, 1, D)
      fus : ViLT multimodal       (B, 1, D)

    Cross-attention fuses [vis; txt] ↔ [fus] through N_CROSS_LAYERS layers.
    Final: concat([sig_pooled, vilt_pooled]) → 2D → classify.
    """
    def __init__(self, siglip_model, vilt_model,
                 d_model=D_MODEL, n_heads=NUM_HEADS,
                 n_layers=N_CROSS_LAYERS, num_classes=NUM_CLASSES,
                 dropout=DROPOUT, unfreeze_n=UNFREEZE_N):
        super().__init__()
        self.sig_enc  = SigLIP2Encoder(siglip_model, d_model, unfreeze_n)
        self.vilt_enc = ViLTEncoder(vilt_model, d_model, unfreeze_n)

        # N bidirectional cross-attention layers
        self.cross_layers = nn.ModuleList([
            CrossAttentionLayer(d_model, n_heads, dropout, DROPPATH_P)
            for _ in range(n_layers)
        ])

        # Final pooling over fused sequences
        self.pool_sig  = AttentionPool(d_model)
        self.pool_vilt = AttentionPool(d_model)

        # Compact classifier (avoids overfitting on small dataset)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout * 0.75),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(d_model // 2, num_classes),
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f'  Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M '
              f'({100*trainable/total:.1f}%)')

    def grouped_params(self):
        bb_names = {'sig_enc.model', 'vilt_enc.model'}
        backbone, head = [], []
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            root = '.'.join(name.split('.')[:2])
            if root in bb_names:
                backbone.append(param)
            else:
                head.append(param)
        return [
            {'params': backbone, 'lr': LR_BACKBONE},
            {'params': head,     'lr': LR_HEAD},
        ]

    def _encode(self, sig_in, vilt_in):
        vis, txt = self.sig_enc(sig_in)     # (B, 1, D) each
        fus      = self.vilt_enc(vilt_in)   # (B, 1, D)

        # Concatenate SigLIP streams into a 2-token sequence
        sig_seq  = torch.cat([vis, txt], dim=1)   # (B, 2, D)
        vilt_seq = fus                            # (B, 1, D)

        for layer in self.cross_layers:
            sig_seq, vilt_seq = layer(sig_seq, vilt_seq)

        sig_pooled  = self.pool_sig(sig_seq)    # (B, D)
        vilt_pooled = self.pool_vilt(vilt_seq)  # (B, D)
        return torch.cat([sig_pooled, vilt_pooled], dim=-1)  # (B, 2D)

    def forward(self, sig_in, vilt_in):
        fused  = self._encode(sig_in, vilt_in)
        return self.classifier(fused)


print('✅ Model architecture ready')

# ── Cell 7: Loss functions ────────────────────────────────────────

class FocalLoss(nn.Module):
    """Focal Loss + label smoothing + class weights."""
    def __init__(self, gamma: float = FOCAL_GAMMA,
                 weight=None, smoothing: float = LABEL_SMOOTH):
        super().__init__()
        self.gamma     = gamma
        self.weight    = weight
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        n       = logits.size(1)
        oh      = torch.zeros_like(logits).scatter(1, targets.unsqueeze(1), 1)
        oh      = oh * (1 - self.smoothing) + self.smoothing / n
        log_p   = F.log_softmax(logits, dim=1)
        if self.weight is not None:
            w  = self.weight[targets].unsqueeze(1)
            ce = -(oh * log_p * w).sum(dim=1)
        else:
            ce = -(oh * log_p).sum(dim=1)
        pt   = torch.exp(-ce.detach())
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


def rdrop_kl(l1: torch.Tensor, l2: torch.Tensor) -> torch.Tensor:
    """Symmetric KL divergence for R-Drop consistency regularisation."""
    p1 = F.log_softmax(l1, dim=-1); q1 = F.softmax(l1, dim=-1)
    p2 = F.log_softmax(l2, dim=-1); q2 = F.softmax(l2, dim=-1)
    return (F.kl_div(p1, q2, reduction='batchmean') +
            F.kl_div(p2, q1, reduction='batchmean')) / 2.0


def mixup_batch(sig_in, vilt_in, labels, alpha=MIXUP_ALPHA):
    """Mixup: interpolate pixel_values only; keep discrete tensors intact.
    Safety: only mix pixel_values that have shape (B, 3, H, W) — float tensors.
    """
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    B   = labels.size(0)
    idx = torch.randperm(B, device=labels.device)

    def _mix(d):
        out = {}
        for k, v in d.items():
            if k == 'pixel_values' and v.is_floating_point() and v.dim() == 4:
                # Ensure both tensors have same shape before mixing
                v_idx = v[idx]
                if v.shape == v_idx.shape:
                    out[k] = lam * v + (1 - lam) * v_idx
                else:
                    out[k] = v   # shapes differ — skip mixup for this key
            else:
                out[k] = v
        return out

    return _mix(sig_in), _mix(vilt_in), labels, labels[idx], lam


class ModelEMA:
    def __init__(self, model, decay: float = EMA_DECAY):
        self.module = copy.deepcopy(model).eval()
        self.decay  = decay

    def update(self, model):
        with torch.no_grad():
            for ep, mp in zip(self.module.parameters(),
                              model.parameters()):
                ep.data.mul_(self.decay).add_(mp.data, alpha=1-self.decay)

print('✅ Loss functions ready')

# ── Cell 8: Train / Eval helpers ─────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, criterion,
                scaler, use_mixup: bool = True, ema=None,
                rdrop_w: float = 0.3):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    optimizer.zero_grad()

    for step, (sig_in, vilt_in, labels) in enumerate(loader):
        sig_in  = {k: v.to(DEVICE) for k, v in sig_in.items()}
        vilt_in = {k: v.to(DEVICE) for k, v in vilt_in.items()}
        labels  = labels.to(DEVICE)

        use_amp_ctx = USE_AMP and DEVICE.type == 'cuda'

        with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp_ctx):
            if use_mixup and random.random() > 0.5:
                sm, vm, ya, yb, lam = mixup_batch(sig_in, vilt_in, labels)
                lg1 = model(sm, vm)
                lg2 = model(sm, vm)   # R-Drop second pass
                ce  = (lam*criterion(lg1,ya)+(1-lam)*criterion(lg1,yb) +
                       lam*criterion(lg2,ya)+(1-lam)*criterion(lg2,yb)) / 2
            else:
                lg1 = model(sig_in, vilt_in)
                lg2 = model(sig_in, vilt_in)
                ce  = (criterion(lg1, labels) + criterion(lg2, labels)) / 2
            loss = (ce + rdrop_kl(lg1, lg2) * rdrop_w) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            if ema is not None:
                ema.update(model)

        total_loss += loss.item() * GRAD_ACCUM * labels.size(0)
        preds_all  += lg1.detach().argmax(1).cpu().tolist()
        labels_all += labels.cpu().tolist()

    n = len(labels_all)
    return total_loss / n, accuracy_score(labels_all, preds_all)


@torch.no_grad()
def eval_epoch(model, loader, criterion, n_tta: int = 1):
    if n_tta > 1:
        model.train()   # keep dropout for TTA
    else:
        model.eval()

    total_loss, logits_all, labels_all = 0.0, [], []
    use_amp_ctx = USE_AMP and DEVICE.type == 'cuda'

    for sig_in, vilt_in, labels in loader:
        sig_in  = {k: v.to(DEVICE) for k, v in sig_in.items()}
        vilt_in = {k: v.to(DEVICE) for k, v in vilt_in.items()}
        labels  = labels.to(DEVICE)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp_ctx):
            if n_tta > 1:
                lg = torch.stack([model(sig_in, vilt_in)
                                   for _ in range(n_tta)]).mean(0)
            else:
                lg = model(sig_in, vilt_in)
            loss = criterion(lg, labels)

        total_loss += loss.item() * labels.size(0)
        logits_all.append(lg.cpu().float())
        labels_all.append(labels.cpu())

    model.eval()
    all_lg = torch.cat(logits_all)
    all_lb = torch.cat(labels_all).numpy()
    preds  = all_lg.argmax(1).numpy()

    return dict(
        loss      = total_loss / len(all_lb),
        acc       = accuracy_score(all_lb, preds),
        macro_f1  = f1_score(all_lb, preds, average='macro',    zero_division=0),
        w_f1      = f1_score(all_lb, preds, average='weighted', zero_division=0),
        prec      = precision_score(all_lb, preds, average='weighted', zero_division=0),
        rec       = recall_score(all_lb, preds, average='weighted', zero_division=0),
    ), preds, all_lb


print('✅ Training helpers ready')

# ── Cell 9: 5-Fold Cross-Validation ──────────────────────────────
print('\n🔄 Loading processors and base models...')
siglip_proc = AutoProcessor.from_pretrained(SIGLIP2_NAME)
vilt_proc   = ViltProcessor.from_pretrained(VILT_NAME)
print('✅ Processors loaded')

labels_arr   = df['label_enc'].values
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

print('\n' + '='*72)
print(f'  SigLIP-2 + ViLT Cross-Attention | 2-Way | {N_FOLDS}-Fold CV')
print(f'  TARGET: 96%+ Accuracy | Overfitting Gap < 0.03')
print('='*72)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(df, labels_arr), 1):
    print(f'\n{"─"*72}')
    print(f'  FOLD {fold} / {N_FOLDS}')
    print(f'{"─"*72}')

    # 80 / 10 / 10 split
    rng   = np.random.default_rng(SEED + fold)
    perm  = rng.permutation(len(train_val_idx))
    val_n = int(round(0.1 * len(df)))
    tr_n  = len(train_val_idx) - val_n

    train_idx = train_val_idx[perm[:tr_n]]
    val_idx   = train_val_idx[perm[tr_n:tr_n + val_n]]

    print(f'  Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

    train_ds = FakedditDataset(df.iloc[train_idx], siglip_proc, vilt_proc, 'train')
    val_ds   = FakedditDataset(df.iloc[val_idx],   siglip_proc, vilt_proc, 'val')
    test_ds  = FakedditDataset(df.iloc[test_idx],  siglip_proc, vilt_proc, 'test')

    nw  = 0  # num_workers=0 avoids DataLoader worker pickling issues on Kaggle
    pin = DEVICE.type == 'cuda'
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=nw, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=nw, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=nw, pin_memory=pin)

    # Fresh model per fold
    siglip_model = AutoModel.from_pretrained(SIGLIP2_NAME)
    vilt_model   = ViltModel.from_pretrained(VILT_NAME)
    model = SigLIPViLTClassifier(
        siglip_model, vilt_model,
        d_model=D_MODEL, n_heads=NUM_HEADS,
        n_layers=N_CROSS_LAYERS, num_classes=NUM_CLASSES,
        dropout=DROPOUT, unfreeze_n=UNFREEZE_N,
    ).to(DEVICE)

    # Class-weighted Focal Loss
    cw = compute_class_weight('balanced',
                              classes=np.arange(NUM_CLASSES),
                              y=labels_arr[train_idx])
    class_weights = torch.tensor(cw, dtype=torch.float32, device=DEVICE)
    criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights,
                          smoothing=LABEL_SMOOTH)

    # Differential LR optimizer
    optimizer = AdamW(model.grouped_params(), weight_decay=WEIGHT_DECAY)

    # Single cosine schedule with warm-up (no LR spikes)
    total_steps  = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
    warmup_steps = int(WARMUP_FRAC * total_steps)
    scheduler    = get_cosine_schedule_with_warmup(
        optimizer, warmup_steps, total_steps)

    # SWA + EMA
    swa_model     = torch.optim.swa_utils.AveragedModel(model)
    swa_scheduler = torch.optim.swa_utils.SWALR(optimizer, swa_lr=LR_BACKBONE)
    swa_start_ep  = max(1, int(SWA_START * NUM_EPOCHS))
    ema           = ModelEMA(model)
    scaler        = torch.amp.GradScaler(device=DEVICE.type, enabled=USE_AMP)

    best_macro_f1 = -1.0
    best_state    = None
    patience_ctr  = 0

    print(f'\n  {"Ep":>3}  {"TrLoss":>8}  {"TrAcc":>7}  '
          f'{"VlAcc":>7}  {"VlMF1":>7}  {"Gap":>8}  Status')
    print(f'  {"─"*3}  {"─"*8}  {"─"*7}  {"─"*7}  {"─"*7}  {"─"*8}  {"─"*10}')

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, scaler,
            use_mixup=(epoch > 2), ema=ema,
        )

        # Evaluate with EMA model
        val_m, _, _ = eval_epoch(ema.module, val_loader, criterion, n_tta=1)
        gap = tr_acc - val_m['acc']

        # SWA update
        if epoch >= swa_start_ep:
            swa_model.update_parameters(model)
            swa_scheduler.step()

        # Early stopping on val macro-F1
        mark = ''
        if val_m['macro_f1'] > best_macro_f1:
            best_macro_f1 = val_m['macro_f1']
            best_state    = {k: v.cpu().clone()
                             for k, v in ema.module.state_dict().items()}
            patience_ctr  = 0
            mark = ' ← ★ best'
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                mark = ' ⏸ stop'

        print(f'  {epoch:>3}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  '
              f'{val_m["acc"]:>7.4f}  {val_m["macro_f1"]:>7.4f}  '
              f'{gap:>+8.4f}{mark}')

        if patience_ctr >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}')
            break

    # Update SWA batch norms
    torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)

    # Restore best EMA checkpoint
    ema.module.load_state_dict(
        {k: v.to(DEVICE) for k, v in best_state.items()})
    test_model = ema.module

    # Train/val metrics for overfitting analysis
    tr_m,   _, _             = eval_epoch(test_model, train_loader, criterion)
    val_m,  _, _             = eval_epoch(test_model, val_loader,   criterion)
    test_m, ts_preds, ts_lbl = eval_epoch(test_model, test_loader,  criterion,
                                           n_tta=TTA_PASSES)

    tv_gap = tr_m['acc'] - val_m['acc']
    tt_gap = tr_m['acc'] - test_m['acc']

    print(f'\n  ── FOLD {fold} TEST RESULTS ──')
    print(f'  Accuracy   : {test_m["acc"]:.4f}')
    print(f'  Macro F1   : {test_m["macro_f1"]:.4f}')
    print(f'  Weighted F1: {test_m["w_f1"]:.4f}')
    print(f'  Precision  : {test_m["prec"]:.4f}')
    print(f'  Recall     : {test_m["rec"]:.4f}')
    print(f'  TV-Gap     : {tv_gap:+.4f} '
          f'{"✅" if abs(tv_gap) < 0.05 else "⚠️"}')
    print(f'  TT-Gap     : {tt_gap:+.4f} '
          f'{"✅" if abs(tt_gap) < 0.05 else "⚠️"}')
    print(f'\n  Classification Report:')
    print(classification_report(ts_lbl, ts_preds,
          target_names=[str(c) for c in le.classes_], zero_division=0))
    print(f'  Confusion Matrix:\n{confusion_matrix(ts_lbl, ts_preds)}')

    fold_results.append({
        'Fold':         fold,
        'Test Acc':     round(test_m['acc'],      4),
        'Macro F1':     round(test_m['macro_f1'], 4),
        'W-F1':         round(test_m['w_f1'],     4),
        'Precision':    round(test_m['prec'],     4),
        'Recall':       round(test_m['rec'],      4),
        'Best Val MF1': round(best_macro_f1,      4),
        'TV-Gap':       round(tv_gap,             4),
        'TT-Gap':       round(tt_gap,             4),
    })

    del model, siglip_model, vilt_model, ema, swa_model
    gc.collect()
    torch.cuda.empty_cache()


# ── Cell 10: Final Summary ────────────────────────────────────────
print('\n' + '='*72)
print('  FINAL 5-FOLD RESULTS — SigLIP-2 + ViLT Cross-Attention')
print('='*72)

res_df = pd.DataFrame(fold_results)
mean_r = res_df.drop(columns=['Fold']).mean().round(4).to_dict()
std_r  = res_df.drop(columns=['Fold']).std().round(4).to_dict()
mean_r['Fold'] = 'Mean'; std_r['Fold'] = 'Std'
summary = pd.concat([res_df,
                     pd.DataFrame([mean_r, std_r])], ignore_index=True)
print(summary.to_string(index=False))
summary.to_csv(f'{OUTPUT_DIR}/fold_results_96pct.csv', index=False)
print(f'\n💾 Saved → {OUTPUT_DIR}/fold_results_96pct.csv')

mean_acc = res_df['Test Acc'].mean()
mean_mf1 = res_df['Macro F1'].mean()
mean_tv  = res_df['TV-Gap'].mean()
mean_tt  = res_df['TT-Gap'].mean()

print(f'\n🏆 Mean Test Accuracy : {mean_acc:.4f}  ({mean_acc*100:.2f}%)')
print(f'🏆 Mean Macro F1     : {mean_mf1:.4f}')
print(f'\n📌 Overfitting:')
print(f'   Mean TV-Gap: {mean_tv:+.4f}')
print(f'   Mean TT-Gap: {mean_tt:+.4f}')
for _, row in res_df.iterrows():
    g = row['TT-Gap']
    st = '✅ Minimal' if abs(g) < 0.03 else ('⚠️  Acceptable' if abs(g) < 0.06 else '❌ High')
    print(f'   Fold {int(row["Fold"])}  TT-Gap={g:+.4f}  {st}')

print(f'\n{"="*72}')
if mean_acc >= 0.96:
    print('  ✅ TARGET ACHIEVED: ≥96% accuracy — ready for academic submission! 🎉')
elif mean_acc >= 0.92:
    print(f'  ⚡ Strong ({mean_acc:.4f}). Increase MAX_SAMPLES or NUM_EPOCHS to reach 96%.')
else:
    print(f'  ⚠️  {mean_acc:.4f}. Check image download success rate and dataset quality.')
print('='*72)

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
Config : epochs=30 | batch=16×4=64 eff | LR bb=5e-06 head=0.0002 | samples=20000
TSV: /kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples/multimodal_train.tsv
Raw rows: 564000 | Cols: ['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label']
Rows with image_url: 562466
Sampled to download: 20000


Images:   0%|          | 0/20000 [00:00<?, ?it/s]

Downloaded: 12841 / 20000
Rows with images: 12841
Classes: [0 1]
Before rebalance:
label_enc
0    9252
1    3589
After rebalance:
label_enc
1    3589
0    3589
Final balanced rows: 7178
✅ Dataset ready
✅ Model architecture ready
✅ Loss functions ready
✅ Training helpers ready

🔄 Loading processors and base models...
✅ Processors loaded

  SigLIP-2 + ViLT Cross-Attention | 2-Way | 5-Fold CV
  TARGET: 96%+ Accuracy | Overfitting Gap < 0.03

────────────────────────────────────────────────────────────────────────
  FOLD 1 / 5
────────────────────────────────────────────────────────────────────────
  Train: 5024 | Val: 718 | Test: 1436


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

...

  Trainable: 106.41M / 335.51M (31.7%)

   Ep    TrLoss    TrAcc    VlAcc    VlMF1       Gap  Status
  ───  ────────  ───────  ───────  ───────  ────────  ──────────
    1    0.1444   0.7110   0.5877   0.5826   +0.1233 ← ★ best
    2    0.1043   0.8414   0.6727   0.6717   +0.1687 ← ★ best
    3    0.1215   0.8434   0.7201   0.7196   +0.1233 ← ★ best
    4    0.1146   0.8392   0.7646   0.7642   +0.0746 ← ★ best
    5    0.1074   0.8462   0.7827   0.7821   +0.0634 ← ★ best
    6    0.1001   0.8533   0.8022   0.8017   +0.0511 ← ★ best
    7    0.0963   0.8565   0.8287   0.8280   +0.0278 ← ★ best
    8    0.0895   0.8886   0.8454   0.8449   +0.0432 ← ★ best
    9    0.0881   0.8724   0.8579   0.8574   +0.0145 ← ★ best
   10    0.0829   0.8888   0.8635   0.8630   +0.0252 ← ★ best
   11    0.0784   0.8872   0.8719   0.8712   +0.0153 ← ★ best
   12    0.0795   0.8732   0.8760   0.8754   -0.0028 ← ★ best
   13    0.0795   0.8826   0.8788   0.8781   +0.0038 ← ★ best
   14    0.0718   0.8983   0

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

ViltModel LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
mlm_score.transform.dense.bias               | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.weight         | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.bias           | UNEXPECTED |  | 
mlm_score.decoder.weight                     | UNEXPECTED |  | 
mlm_score.bias                               | UNEXPECTED |  | 
mlm_score.transform.dense.weight             | UNEXPECTED |  | 
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Trainable: 106.41M / 335.51M (31.7%)

   Ep    TrLoss    TrAcc    VlAcc    VlMF1       Gap  Status
  ───  ────────  ───────  ───────  ───────  ────────  ──────────
    1    0.1457   0.7003   0.4526   0.4501   +0.2477 ← ★ best
    2    0.1037   0.8482   0.6699   0.6699   +0.1782 ← ★ best
    3    0.1173   0.8374   0.8022   0.8017   +0.0352 ← ★ best
    4    0.1101   0.8442   0.8259   0.8251   +0.0183 ← ★ best
    5    0.1099   0.8424   0.8357   0.8348   +0.0067 ← ★ best
    6    0.0986   0.8551   0.8496   0.8487   +0.0055 ← ★ best
    7    0.0940   0.8617   0.8538   0.8528   +0.0079 ← ★ best
    8    0.0926   0.8665   0.8649   0.8640   +0.0016 ← ★ best
    9    0.0826   0.8655   0.8691   0.8682   -0.0036 ← ★ best
   10    0.0799   0.8856   0.8760   0.8753   +0.0095 ← ★ best
   11    0.0761   0.8854   0.8774   0.8767   +0.0079 ← ★ best
   12    0.0686   0.9154   0.8774   0.8767   +0.0380
   13    0.0739   0.8802   0.8844   0.8837   -0.0042 ← ★ best
   14    0.0753   0.8746   0.8802   0

In [ ]:
SigLIP-2 + ViLT Cross-Attention Fusion — Fakeddit 2-Way

Performance Summary by Fold (Best Epoch)
Fold	Best Epoch	Validation Accuracy	Validation Macro F1
Fold 1	16	0.8969	0.8969
Fold 2	17	0.8955	0.8951
Fold 3	18	0.8830	0.8829
Fold 4	17	0.8858	0.8851
Fold 5	20	0.8928	0.8922
Average	—	0.8908	0.8904
